# Simulated population using best-inferred parameters

This notebook reproduces Figures 8 and 9 in Pardo et al. (2025).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import mlpoppyns.generator.maps.axes_scaling as axs
import utilities.plot_settings
import mlpoppyns.simulator.basics.constants as const
from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 18
MEDIUM_SIZE = 28
BIGGER_SIZE = 30

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

## Load the simulation 

Import three simulated surveys.

In [ ]:
directory = "../../data/paper_results/pardo_etal_2025/best_inferred_atnf"
df_PMPS_sim = pd.read_pickle(
    directory + f"/survey_PMPS_results.pkl.gz",
    compression="gzip",
)

df_SMPS_sim = pd.read_pickle(
    directory + f"/survey_SMPS_results.pkl.gz",
    compression="gzip",
)

df_HTRU_sim = pd.read_pickle(
    directory + f"/survey_HTRU_low_mid_results.pkl.gz",
    compression="gzip",
)

df_PMPS_sim.head()

In [ ]:
# Extracting the parameters.
RA_pmps_sim = df_PMPS_sim["RA"]["[deg]"].to_numpy()
DEC_pmps_sim = df_PMPS_sim["DEC"]["[deg]"].to_numpy()
l_pmps_sim = df_PMPS_sim["l"]["[deg]"].to_numpy()
b_pmps_sim = df_PMPS_sim["b"]["[deg]"].to_numpy()
pmRA_pmps_sim = df_PMPS_sim["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_pmps_sim = df_PMPS_sim["pm_DEC"]["[mas yr^-1]"].to_numpy()
DM_pmps_sim = df_PMPS_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_pmps_sim = df_PMPS_sim["d"]["[kpc]"].to_numpy()
P_pmps_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
Pdot_pmps_sim = df_PMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_pmps_sim = df_PMPS_sim["S_radio_obs_mean_1400"]["[Jy]"].to_numpy()
w_eff_pmps_sim = df_PMPS_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_smps_sim = df_SMPS_sim["RA"]["[deg]"].to_numpy()
DEC_smps_sim = df_SMPS_sim["DEC"]["[deg]"].to_numpy()
l_smps_sim = df_SMPS_sim["l"]["[deg]"].to_numpy()
b_smps_sim = df_SMPS_sim["b"]["[deg]"].to_numpy()
pmRA_smps_sim = df_SMPS_sim["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_smps_sim = df_SMPS_sim["pm_DEC"]["[mas yr^-1]"].to_numpy()
DM_smps_sim = df_SMPS_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_smps_sim = df_SMPS_sim["d"]["[kpc]"].to_numpy()
P_smps_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
Pdot_smps_sim = df_SMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_smps_sim = df_SMPS_sim["S_radio_obs_mean_1400"]["[Jy]"].to_numpy()
w_eff_smps_sim = df_SMPS_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_htru_sim = df_HTRU_sim["RA"]["[deg]"].to_numpy()
DEC_htru_sim = df_HTRU_sim["DEC"]["[deg]"].to_numpy()
l_htru_sim = df_HTRU_sim["l"]["[deg]"].to_numpy()
b_htru_sim = df_HTRU_sim["b"]["[deg]"].to_numpy()
pmRA_htru_sim = df_HTRU_sim["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_htru_sim = df_HTRU_sim["pm_DEC"]["[mas yr^-1]"].to_numpy()
DM_htru_sim = df_HTRU_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_htru_sim = df_HTRU_sim["d"]["[kpc]"].to_numpy()
P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
Pdot_htru_sim = df_HTRU_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_htru_sim = df_HTRU_sim["S_radio_obs_mean_1400"]["[Jy]"].to_numpy()
w_eff_htru_sim = df_HTRU_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
colors = ["#FFAC1C", "dodgerblue", "#440154"]

$\dot{E}_{\rm rot}$ lines.

In [ ]:
R_NS = cfg["NS_radius"]
mass_NS = cfg["NS_mass"]
I_NS = 2.0 / 5.0 * mass_NS * R_NS**2

Pdot_Edot_lines = np.zeros((6, 51))

Edot_log = np.linspace(28, 38, 6)
print(Edot_log)

In [ ]:
P_log = np.linspace(-3, 2, 51)

In [ ]:
for i in range(len(Edot_log)):
    Pdot_Edot_lines[i] = (
        10 ** Edot_log[i] * (10**P_log) ** 3 / (4 * np.pi**2 * I_NS)
    )

B lines.

In [ ]:
Pdot_B_lines = np.zeros((5, 51))

B_log = np.linspace(10, 14, 5)
print(B_log)

In [ ]:
for i in range(len(B_log)):
    Pdot_B_lines[i] = (
        np.pi**2
        * (10 ** B_log[i]) ** 2
        * (R_NS**6)
        / (I_NS * 10**P_log * const.C**3)
    )

## Load the observed population

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.columns = df_atnf.columns.droplevel(1)
df_atnf.head()

In [ ]:
df_meerkat = df_meerkat_1 = pd.read_csv(
    "../../data/observations/meerkat_tpa_posselt_2023.csv",
    delimiter=",",
)

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PMRA",
        "PMDEC",
        "PX",
        "POSEPOCH",
        "RAJD",
        "DECJD",
        "DM",
        "W50",
        "W10",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "DIST_DM",
        "ZZ",
        "XX",
        "YY",
        "Unnamed: 2_level_0",
        "Unnamed: 4_level_0",
        "Unnamed: 6_level_0",
        "Unnamed: 7_level_0",
        "Unnamed: 51_level_0"
    ],
)

len(df_atnf)

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"].isin(["NAN"])]

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf["ASSOC"].str.match("|".join(discard))
]

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]

In [ ]:
# Parks multibeam pulsar survey database.
df_atnf_pmps = df_atnf[
    df_atnf["SURVEY"].str.contains("pksmb")
]
l_pmps_obs = df_atnf_pmps["Gl"].to_numpy().astype(np.float64)
b_pmps_obs = df_atnf_pmps["Gb"].to_numpy().astype(np.float64)
P_pmps_obs = df_atnf_pmps["P0"].to_numpy().astype(np.float64)
Pdot_pmps_obs = df_atnf_pmps["P1"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] = (
    l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] - 360.0
)

# Select only pulsars falling in the PMPS sky coverage where completness is above 90%.
cond = (l_pmps_obs > -100.0) & (l_pmps_obs < 50.0) & (np.abs(b_pmps_obs) < 5.0)

# Merge the MeerKAT TPA program data with the PMPS ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements
# for the PMPS pulsars.
df_meerkat_pmps = pd.merge(
    df_meerkat, df_atnf_pmps[cond], left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_pmps = df_meerkat_pmps.dropna(subset=["ch6flux"])

S1400_pmps_obs = df_meerkat_pmps['ch6flux'].to_numpy().astype(np.float64)

# Swinburne multibeam pulsar survey database.
df_atnf_smps = df_atnf[
    df_atnf["SURVEY"].str.contains("pkssw")
]

l_smps_obs = df_atnf_smps["Gl"].to_numpy().astype(np.float64)
b_smps_obs = df_atnf_smps["Gb"].to_numpy().astype(np.float64)
P_smps_obs = df_atnf_smps["P0"].to_numpy().astype(np.float64)
Pdot_smps_obs = df_atnf_smps["P1"].to_numpy().astype(np.float64)
# Convert galactic latitude in the range [-180., 180].
l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] = (
    l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the SMPS sky coverage where completness is above 90%.
cond = (l_smps_obs > -100.0) & (l_smps_obs < 50.0)

# Merge the MeerKAT TPA program data with the SMPS ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements
# for the SMPS pulsars.
df_meerkat_smps = pd.merge(
    df_meerkat, df_atnf_smps[cond], left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_smps = df_meerkat_smps.dropna(subset=["ch6flux"])
S1400_smps_obs = df_meerkat_smps["ch6flux"].to_numpy().astype(np.float64)


# HTRU multibeam pulsar survey database.
df_atnf_htru = df_atnf[
    df_atnf["SURVEY"].str.contains("htru_pks")
]

l_htru_obs = df_atnf_htru["Gl"].to_numpy().astype(np.float64)
b_htru_obs = df_atnf_htru["Gb"].to_numpy().astype(np.float64)
P_htru_obs = df_atnf_htru["P0"].to_numpy().astype(np.float64)
Pdot_htru_obs = df_atnf_htru["P1"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] = (
    l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the HTRU sky coverage where completness is above 90%.
cond = (
    (l_htru_obs > -120.0) & (l_htru_obs < 30.0) & (np.abs(b_htru_obs) < 15.0)
)

# Merge the MeerKAT TPA program data with the HTRU ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements
# for the HTRU pulsars.
df_meerkat_htru = pd.merge(
    df_meerkat, df_atnf_htru[cond], left_on="PSRJ", right_on="PSRJ"
)

df_meerkat_htru = df_meerkat_htru.dropna(subset=["ch6flux"])
S1400_htru_obs = df_meerkat_htru["ch6flux"].to_numpy().astype(np.float64)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7), sharey=True)

# Titles for each plot.
titles = ["PMPS", "SMPS", "HTRU"]

# Data for each plot.
datasets = [
    (P_pmps_sim, Pdot_pmps_sim, P_pmps_obs, Pdot_pmps_obs),
    (P_smps_sim, Pdot_smps_sim, P_smps_obs, Pdot_smps_obs),
    (P_htru_sim, Pdot_htru_sim, P_htru_obs, Pdot_htru_obs),
]

# Annotations for Edot and B field lines.
Edot_annotations = [
    (0.41, 0.015, r"$10^{28} \, {\rm erg} \, {\rm s}^{-1}$"),
    (0.275, 0.015, r"$10^{30} \, {\rm erg} \, {\rm s}^{-1}$"),
    (0.143, 0.015, r"$10^{32} \, {\rm erg} \, {\rm s}^{-1}$"),
    (0.01, 0.015, r"$10^{34} \, {\rm erg} \, {\rm s}^{-1}$"),
    (0.01, 0.181, r"$10^{36} \, {\rm erg} \, {\rm s}^{-1}$"),
    (0.01, 0.347, r"$10^{38} \, {\rm erg} \, {\rm s}^{-1}$"),
]
B_field_annotations = [
    (0.89, 0.005, r"$10^{10} \, {\rm G}$"),
    (0.89, 0.172, r"$10^{11} \, {\rm G}$"),
    (0.89, 0.340, r"$10^{12} \, {\rm G}$"),
    (0.89, 0.506, r"$10^{13} \, {\rm G}$"),
    (0.89, 0.672, r"$10^{14} \, {\rm G}$"),
]

# Iterate over the datasets and corresponding axes.
for ax, (P_sim, Pdot_sim, P_obs, Pdot_obs), title in zip(axes, datasets, titles):
    # Plot the Edot and B field lines.
    for i in range(len(Edot_log)):
        ax.plot(
            10**P_log,
            Pdot_Edot_lines[i],
            linestyle="-",
            color="gray",
            alpha=0.5,
            rasterized=True,
        )
    for i in range(len(B_log)):
        ax.plot(
            10**P_log,
            Pdot_B_lines[i],
            linestyle="-",
            color="gray",
            alpha=0.5,
            rasterized=True,
        )

    # Plot simulated and observed data for each survey.
    ax.plot(
        P_sim,
        Pdot_sim,
        linestyle="None",
        marker="o",
        color=colors[1],
        markersize=9,
        alpha=1.0,
        rasterized=True,
        label="Simulated",
    )
    ax.plot(
        P_obs,
        Pdot_obs,
        linestyle="None",
        marker="*",
        color=colors[0],
        markersize=9,
        alpha=1.0,
        rasterized=True,
        label="Observed",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1.0e-3, 100.0)
    ax.set_ylim(1.0e-21, 1.0e-9)

    # Add Edot and B field annotations
    for x, y, text in Edot_annotations:
        ax.text(
            x,
            y,
            text,
            rotation=50,
            fontsize=14,
            transform=ax.transAxes,
            color="gray",
        )

    for x, y, text in B_field_annotations:
        ax.text(
            x,
            y,
            text,
            rotation=-23,
            fontsize=14,
            transform=ax.transAxes,
            color="gray",
        )

    ax.set_title(title, fontsize=30)

# Add legends to the first plot.
axes[0].legend(frameon=True, loc=2, fontsize=20)

# Remove y-axis from the middle and right plots
axes[1].tick_params(labelleft=False)
axes[2].tick_params(labelleft=False)

# Add y-axis label only to the left plot
axes[0].set_ylabel(r"Period derivative $\dot{P}$ [s$\,{\rm s}^{-1}$]")

# Add x-axis label to all plots
for ax in axes:
    ax.set_xlabel(r"Period $P$ [s]")

plt.subplots_adjust(wspace=0.05)

# Adjust layout and save the figure
plt.tight_layout()
plt.savefig(
    f"../../paper_plots/pardo_et_al_2025/plots/comparison_ppdot_combined.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Plotting the simulated and observed fluxes at 1.4 GHz measured in mJy.

In [ ]:
# New X-axis values.
x = np.linspace(-2, 3.5, 1000)

# Estimate the PDF using a Gaussian kernel.
kde_pmps_sim = stats.gaussian_kde(np.log10(S1400_pmps_sim * 1e3))

kde_smps_sim = stats.gaussian_kde(np.log10(S1400_smps_sim * 1e3))

kde_htru_sim = stats.gaussian_kde(np.log10(S1400_htru_sim * 1e3))

In [ ]:
# Remove NaNs and estimate the PDF using a Gaussian kernel.
S1400_pmps_obs = S1400_pmps_obs[~np.isnan(S1400_pmps_obs)]
kde_pmps_obs = stats.gaussian_kde(np.log10(S1400_pmps_obs))

S1400_smps_obs = S1400_smps_obs[~np.isnan(S1400_smps_obs)]
kde_smps_obs = stats.gaussian_kde(np.log10(S1400_smps_obs))

S1400_htru_obs = S1400_htru_obs[~np.isnan(S1400_htru_obs)]
kde_htru_obs = stats.gaussian_kde(np.log10(S1400_htru_obs))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

# KDE density plots simulated.
ax.plot(
    x,
    kde_pmps_sim(x),
    color=colors[0],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE simulated PMPS",
)
ax.plot(
    x,
    kde_smps_sim(x),
    color=colors[1],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE simulated SMPS",
)
ax.plot(
    x,
    kde_htru_sim(x),
    color=colors[2],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE simulated HTRU",
)

# KDE density plots ATNF populations.
ax.plot(
    x,
    kde_pmps_obs(x),
    color=colors[0],
    alpha=0.5,
    linewidth=5,
    label=r"KDE observed PMPS",
)
ax.plot(
    x,
    kde_smps_obs(x),
    color=colors[1],
    alpha=0.5,
    linewidth=5,
    label=r"KDE observed SMPS",
)
ax.plot(
    x,
    kde_htru_obs(x),
    color=colors[2],
    alpha=0.5,
    linewidth=5,
    label=r"KDE observed HTRU",
)

ax.set_xlim(-2, 3.5)
ax.set_ylim(0.0, 1.0)
ax.set_xlabel(r"Mean flux density log$_{10}$ $S_{{\rm mean}, 1400}$ [mJy]")
ax.set_ylabel(r"Normalized pulsar count")
ax.legend(frameon=True, loc="best")

plt.tight_layout()
plt.savefig(
    f"../../paper_plots/pardo_et_al_2025/plots/KDE_comparison.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()